# RAG v1 vs v2: experiment design

Notebook 01 explored retrieval options for v1 (OpenSearch) with no constraints. This notebook turns the v1 vs v2 comparison into a **controlled experiment**. It fixes the RAG parameters that **both** AWS services can implement identically, so the only thing that differs between the variants is the service itself.

By the end it writes `evals/rag/experiment_config.json`, the frozen spec both variants are built from on Day 2–3.

| Part | What | Needs |
| --- | --- | --- |
| 0 | The language of experimentation, applied to this project | nothing |
| 1 | What each service supports, and what they have in common | nothing |
| 2 | Setup (loaders, chunkers, eval set, embeddings; copied from 01 so this notebook stands alone) | Bedrock for uncached embeddings |
| 3 | Chunking strategy, restricted to options both services can run | Bedrock |
| 4 | Embedding model and dimensions | Bedrock |
| 5 | Retrieval mode | nothing |
| 6 | top-k | Bedrock |
| 7 | What still differs between the arms, and the parity check | nothing |
| 8 | Freeze the spec | nothing |

**Run it:** `uv run --group notebook jupyter lab notebooks/`. Embeddings are shared with notebook 01 through `notebooks/.cache/embeddings.json`, so most of them are already cached.

## Part 0. The language of experimentation

| Term | Meaning | In this experiment |
| --- | --- | --- |
| **Independent variable** (or *factor*) | The one thing you change on purpose | The retrieval **platform**: OpenSearch with our own ingestion pipeline vs. a Bedrock Knowledge Base |
| **Levels / arms / treatments** | The values the factor takes | `v1` (OpenSearch), `v2` (Bedrock KB) |
| **Controlled variables** | Everything held fixed, so it can't explain a difference | Source documents, chunks, embedding model and dimensions, retrieval mode, distance metric, top-k, eval questions, and the LLM and prompt used to generate answers |
| **Dependent variables** (or *outcomes*, *metrics*) | What you measure | hit@k, MRR, recall@k, faithfulness and relevance, p50 and p95 latency, cost |
| **Confounder** | An uncontrolled difference that changes together with the factor, so you can't tell which one caused an effect | e.g. v2 cuts 300-token windows while v1 cuts by markdown section. A quality gap could then come from the chunking, not the platform |
| **Baseline** | The reference arm the other is compared against | `v1`: we built it and understand every knob |
| **Hypothesis** | The prediction you write down **before** running | See below |
| **Decision rule** | How the winner is chosen, fixed in advance | In `docs/architecture.md` (quality first, then cost, then latency) |
| **Parity check** (A/A-style sanity check) | Running both arms where they *should* agree, to catch setup bugs before measuring | Do the two arms return the same top-k chunk IDs? (Part 7) |
| **Noise** | Variation that is not caused by the factor | 18 eval questions means one question moves a metric by 0.056. Latency varies from run to run |

### What controlling everything implies

Once chunks, embeddings, mode and k are the same, both arms run **cosine similarity over the same vectors**, so retrieval quality should come out (almost) the **same**. That is the expected result, not a failed experiment. It tells us that the real differences between the platforms are:

- **latency** (network path, API overhead, vector index implementation)
- **cost** (fixed monthly cost vs. pay per use)
- **operational effort** (who runs ingestion, what can break)

…plus small quality effects from approximate nearest-neighbour search. A **large** quality gap would point to a parity bug (e.g. the KB stores different text than we think), and the parity check exists to catch it.

**Hypothesis (write it before Day 3):**
> With identical chunks, embeddings, retrieval mode and top-k, v1 and v2 reach the same retrieval quality within noise (|ΔMRR| < 0.056). v2 costs less per month at our traffic and needs less operational work. Their latencies differ by less than 100 ms at p50.

### A different question you might also want answered

"Which service is better when each is tuned to its best?" (v1 with hybrid search and custom chunking, v2 with semantic chunking and its reranker) is a **system comparison**: many variables change at once, so you learn *which package* wins but not *why*. It is a valid follow-up, but it is not this experiment.

## Part 1. What each service supports

The shared parameters have to come from the intersection of what both services can do.

| Parameter | v1: OpenSearch + our pipeline | v2: Bedrock Knowledge Base | Common ground |
| --- | --- | --- | --- |
| **Chunking** | Anything; we write the code | Default (~300 tokens), fixed-size (tokens + overlap %), hierarchical, semantic, **`NONE` (each file = one chunk)**, or a custom transformation Lambda | **Our own chunker → one S3 file per chunk → KB with `chunkingStrategy: NONE`.** Both arms index exactly the same chunk text |
| **Chunk metadata** | Any fields in the index | A `<file>.metadata.json` sidecar per file (≤ 10 KB) | `doc_id`, `chunk_id`, `title`, `section` in both |
| **Embedding model** | Any Bedrock model; we call it | Titan Text Embeddings V2 (256 / 512 / 1024 dims), Cohere Embed, … | **Titan Text Embeddings V2**, same dims |
| **Retrieval mode** | BM25, k-NN, hybrid with configurable normalisation, weights or RRF | `SEMANTIC` or `HYBRID` (`overrideSearchType`). Hybrid only works on some vector stores (OpenSearch Serverless, RDS, MongoDB), and the score fusion is **not configurable** | **Semantic (k-NN)**. Hybrid can't be made identical: v2's fusion is a black box |
| **Distance** | `cosinesimil` | Set on the vector store (S3 Vectors: cosine or euclidean) | **Cosine** |
| **top-k** | `size` / `k` | `numberOfResults` | Same k |
| **Reranking** | Optional | Optional Bedrock rerank model | **Off** in both |

### 1b. v2's vector store decides the retrieval mode

A Bedrock KB doesn't store vectors itself. You attach a vector store, and that choice matters for the experiment:

| v2 vector store | Cost profile | Hybrid in KB | Effect on the experiment |
| --- | --- | --- | --- |
| **Amazon S3 Vectors** | Pay per use, near zero when idle | No (semantic only) | A genuinely different backend from v1. **Recommended** |
| OpenSearch Serverless | Minimum OCUs billed even when idle (roughly $175+/month) | Yes | v2 would also run on OpenSearch, so the factor shrinks to "who runs the ingestion pipeline". Expensive to leave running |
| OpenSearch managed cluster | Same as v1 | Not confirmed | Both arms on the same engine. The comparison mostly disappears |

**Recommendation:** use **S3 Vectors** for v2. That fixes the retrieval mode to **semantic** for both arms, which Part 5 shows costs us little on this corpus.

Sources: [Bedrock KB chunking (`NONE`)](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_agent_ChunkingConfiguration.html), [KB query configuration and hybrid support](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-test-config.html), [S3 Vectors with Bedrock KB](https://docs.aws.amazon.com/AmazonS3/latest/userguide/s3-vectors-bedrock-kb.html), [custom chunking with `NONE`](https://aws.amazon.com/blogs/machine-learning/accelerate-performance-using-a-custom-chunking-mechanism-with-amazon-bedrock/). Before building v2, check the S3 Vectors metadata size limits against the largest chunk (Part 3 prints it).

## Part 2. Setup

The loaders, chunkers, eval set and embedding cache are copied from notebook 01. Chunk text has to be **byte-identical** to 01 for the embedding cache to hit, so change them in both notebooks or not at all.

In [1]:
import hashlib
import json
import re
import sys
from datetime import date
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
from langchain_text_splitters import MarkdownHeaderTextSplitter

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
import config

KB_DIR = ROOT / "data" / "knowledge_base"
CACHE_DIR = ROOT / "notebooks" / ".cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
REGION = config.get_aws_region()
pd.set_option("display.max_colwidth", 120)


def load_documents(kb_dir: Path) -> list[dict]:
    docs = []
    for path in sorted(kb_dir.rglob("*.md")):
        text = path.read_text(encoding="utf-8")
        title = next((l[2:].strip() for l in text.splitlines() if l.startswith("# ")), path.stem)
        docs.append({"doc_id": path.stem, "title": title, "text": text})
    return docs


def approx_tokens(text: str) -> int:
    return round(len(text) / 4)


docs = load_documents(KB_DIR)
print(f"{len(docs)} documents, region {REGION}")

11 documents, region us-east-1


In [2]:
# --- chunkers (same as notebook 01) ---
def make_chunk(doc, i, text, section=""):
    return {"chunk_id": f"{doc['doc_id']}#{i}", "doc_id": doc["doc_id"], "title": doc["title"],
            "section": section, "text": text.strip()}


def chunk_fixed(doc, size_words=100, overlap_words=20):
    words = doc["text"].split()
    step = size_words - overlap_words
    starts = range(0, max(len(words) - overlap_words, 1), step)
    return [make_chunk(doc, i, " ".join(words[s:s + size_words])) for i, s in enumerate(starts)]


_md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "title"), ("##", "section")], strip_headers=False)


def chunk_markdown(doc, add_context=False):
    chunks = []
    for i, part in enumerate(_md_splitter.split_text(doc["text"])):
        section = part.metadata.get("section", "")
        text = part.page_content
        if add_context and section:
            text = f"{doc['title']} | {section}\n{text}"
        chunks.append(make_chunk(doc, i, text, section))
    return chunks


QA_PATTERN = re.compile(r"^\*\*(.+\?)\*\*\s*$", re.M)


def chunk_qa_aware(doc):
    pieces = []
    for part in _md_splitter.split_text(doc["text"]):
        section = part.metadata.get("section", "")
        body = part.page_content
        prefix = f"{doc['title']} | {section}\n" if section else ""
        qs = list(QA_PATTERN.finditer(body))
        if len(qs) >= 2:
            for j, m in enumerate(qs):
                end = qs[j + 1].start() if j + 1 < len(qs) else len(body)
                pieces.append((section, prefix + body[m.start():end].strip()))
        else:
            pieces.append((section, prefix + body))
    return [make_chunk(doc, i, t, s) for i, (s, t) in enumerate(pieces)]

In [3]:
# --- throwaway eval set (same as notebook 01; the Day 3 golden set replaces it) ---
EVAL_SET = [
    {"q": "Is the bond oil safe for bleached hair?", "relevant": ["repair-bond-oil"], "type": "lexical"},
    {"q": "How much does shipping cost?", "relevant": ["faq"], "type": "lexical"},
    {"q": "Up to what temperature does the heat protectant work?", "relevant": ["heat-shield-mist"], "type": "lexical"},
    {"q": "Is niacinamide in any of your products?", "relevant": ["scalp-reset-serum"], "type": "lexical"},
    {"q": "Does the conditioner contain silicones?", "relevant": ["silk-slip-conditioner"], "type": "lexical"},
    {"q": "Is there a holiday sale?", "relevant": ["aws-developers-discount"], "type": "lexical"},
    {"q": "Are your products tested on animals?", "relevant": ["faq"], "type": "lexical"},
    {"q": "How often should I use the deep conditioning mask?", "relevant": ["weekly-repair-mask"], "type": "lexical"},
    {"q": "What routine do you recommend for fine hair?", "relevant": ["hair-type-guide"], "type": "lexical"},
    {"q": "What should I spray on before using my flat iron?", "relevant": ["heat-shield-mist"], "type": "paraphrase"},
    {"q": "My scalp gets greasy really fast, what can help?", "relevant": ["scalp-reset-serum", "hair-type-guide"], "type": "paraphrase"},
    {"q": "Do you have any promo codes?", "relevant": ["aws-developers-discount"], "type": "paraphrase"},
    {"q": "Which product gives my curls hold without making them stiff?", "relevant": ["air-dry-styling-gel", "define-curl-cream"], "type": "paraphrase"},
    {"q": "Can I get my money back if I already opened the bottle?", "relevant": ["faq"], "type": "paraphrase"},
    {"q": "How many days do I have to send something back?", "relevant": ["faq"], "type": "paraphrase"},
    {"q": "Which shampoo won't wash out my dye job?", "relevant": ["hydra-wash-shampoo"], "type": "paraphrase"},
    {"q": "When will I see the money from my return on my card?", "relevant": ["faq"], "type": "paraphrase"},
    {"q": "Will the product with argan oil make thin hair look oily?", "relevant": ["repair-bond-oil"], "type": "paraphrase"},
]
NOISE = round(1 / len(EVAL_SET), 3)  # one question's worth of a metric


def run(search_fn, k: int = 10) -> list[dict]:
    """Run every eval question once and keep the full ranking, so metrics can be read at any k."""
    rows = []
    for item in EVAL_SET:
        hits = search_fn(item["q"], k)
        rows.append({"q": item["q"], "type": item["type"], "relevant": set(item["relevant"]),
                     "ranking": [h["doc_id"] for h in hits], "tokens": [approx_tokens(h["text"]) for h in hits]})
    return rows


def metrics_at(rows: list[dict], k: int) -> pd.Series:
    """hit@1, hit@k, recall@k (share of relevant docs in the top k), MRR (within the full ranking), tokens in the top k."""
    out = []
    for r in rows:
        first = next((i + 1 for i, d in enumerate(r["ranking"]) if d in r["relevant"]), None)
        out.append({
            "hit@1": float(first == 1),
            "hit@k": float(first is not None and first <= k),
            "recall@k": len(r["relevant"] & set(r["ranking"][:k])) / len(r["relevant"]),
            "mrr": 1 / first if first else 0.0,
            "ctx_tokens@k": sum(r["tokens"][:k]),
        })
    return pd.DataFrame(out).mean().round(3)


print(f"{len(EVAL_SET)} questions; one question = {NOISE} of a metric")

18 questions; one question = 0.056 of a metric


In [4]:
# --- embeddings: Titan Text Embeddings V2, cached (same cache key as notebook 01) ---
EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"
bedrock = boto3.client("bedrock-runtime", region_name=REGION)
_cache_path = CACHE_DIR / "embeddings.json"
_cache = json.loads(_cache_path.read_text()) if _cache_path.exists() else {}


def embed(text: str, dims: int) -> list[float]:
    key = hashlib.sha256(f"{EMBED_MODEL_ID}|{dims}|{text}".encode()).hexdigest()
    if key not in _cache:
        body = json.dumps({"inputText": text, "dimensions": dims, "normalize": True})
        _cache[key] = json.loads(bedrock.invoke_model(modelId=EMBED_MODEL_ID, body=body)["body"].read())["embedding"]
    return _cache[key]


class SemanticIndex:
    """Exact cosine k-NN over one chunk set. Both arms approximate this; OpenSearch and S3 Vectors use ANN indexes."""

    def __init__(self, chunks: list[dict], dims: int = 1024):
        self.chunks, self.dims = chunks, dims
        self.emb = np.array([embed(c["text"], dims) for c in chunks], dtype=np.float32)
        _cache_path.write_text(json.dumps(_cache))

    def search(self, q: str, k: int = 10) -> list[dict]:
        scores = self.emb @ np.array(embed(q, self.dims), dtype=np.float32)
        return [{**self.chunks[i], "score": float(scores[i])} for i in np.argsort(-scores, kind="stable")[:k]]

## Part 3. Chunking strategy

**Constraint:** v2 has to receive the same chunks as v1, so chunking happens in **our code**, and the KB ingests one file per chunk with `chunkingStrategy: NONE`. KB's built-in chunkers are out, because we can't reproduce their exact boundaries in v1 (the tokenizer and sentence-boundary rules aren't published).

**Candidates** (all from notebook 01, plus one reference):

| Name | What | Why it's here |
| --- | --- | --- |
| `kb_fixed_300t~` | ~225-word windows with 20% overlap, an approximation of KB's default 300-token chunking | Reference: roughly what v2 would do if we left chunking to it. It shows what we give up or gain by controlling chunking |
| `markdown` | One chunk per `##` section | Best semantic result in 01 |
| `markdown_ctx` | `markdown` + a `"<title> \| <section>"` prefix | Chunk names its product |
| `qa_aware` | `markdown_ctx`, with Q&A sections split per question | Fewest tokens per chunk |

**Decision rule (fixed before looking at the table):**
1. Retrieval mode is semantic (Part 1b), so compare semantic results only.
2. Highest **MRR** wins. Differences smaller than one question (0.056) count as a **tie**.
3. Tie-breakers, in order: higher **recall@3**, then fewer **ctx_tokens@3**.

In [5]:
CANDIDATES = {
    "kb_fixed_300t~": lambda d: chunk_fixed(d, 225, 45),
    "markdown": chunk_markdown,
    "markdown_ctx": lambda d: chunk_markdown(d, add_context=True),
    "qa_aware": chunk_qa_aware,
}
chunk_sets = {name: [c for d in docs for c in fn(d)] for name, fn in CANDIDATES.items()}

pd.DataFrame({
    name: {
        "chunks": len(cs),
        "mean_tokens": round(np.mean([approx_tokens(c["text"]) for c in cs])),
        "max_tokens": max(approx_tokens(c["text"]) for c in cs),
        "max_bytes": max(len(c["text"].encode()) for c in cs),  # check against S3 Vectors / KB metadata limits
    }
    for name, cs in chunk_sets.items()
}).T

,chunks,mean_tokens,max_tokens,max_bytes
kb_fixed_300t~,19,271,357,1431
markdown,76,61,326,1312
markdown_ctx,76,70,345,1387
qa_aware,105,54,222,890


In [6]:
DIMS_DEFAULT = 1024
indexes = {name: SemanticIndex(cs, DIMS_DEFAULT) for name, cs in chunk_sets.items()}
runs = {name: run(idx.search) for name, idx in indexes.items()}

chunking_table = pd.DataFrame({name: metrics_at(rows, 3) for name, rows in runs.items()}).T
chunking_table.columns = [c.replace("@k", "@3") for c in chunking_table.columns]
chunking_table.sort_values("mrr", ascending=False)

,hit@1,hit@3,recall@3,mrr,ctx_tokens@3
markdown,1.000,1.000,1.000,1.000,248.500
qa_aware,0.944,0.944,0.917,0.944,153.889
markdown_ctx,0.889,0.944,0.917,0.917,276.778
kb_fixed_300t~,0.722,1.000,1.000,0.852,813.500


In [7]:
# Where does each strategy lose points? Lexical vs paraphrase MRR.
pd.DataFrame({
    name: pd.DataFrame([{"type": r["type"], "mrr": metrics_at([r], 10)["mrr"]} for r in rows]).groupby("type")["mrr"].mean()
    for name, rows in runs.items()
}).T.round(3)

type,lexical,paraphrase
kb_fixed_300t~,0.815,0.889
markdown,1.000,1.000
markdown_ctx,0.889,0.944
qa_aware,0.889,1.000


In [8]:
CHUNKING = "markdown"  # <- apply the decision rule to the table above
print(f"chosen: {CHUNKING}  ({len(chunk_sets[CHUNKING])} chunks)")

chosen: markdown  (76 chunks)


**Reading the result**

- Section-based chunking beats the KB-style fixed windows. Fixed windows reach the right document by rank 3, but their MRR is lower and they put ~3× more tokens in the prompt. Controlling chunking costs us nothing; it is also the better strategy.
- `markdown` scores higher MRR than `markdown_ctx` in semantic mode, but its chunks don't name their product (a "How to use" chunk doesn't say *which* product). That is a **generation** problem, not a retrieval one. Fix it without changing what gets embedded: store `title` and `section` as metadata (index fields in v1, the `.metadata.json` sidecar in v2), and have `rag_client` prepend `"<title> | <section>"` when it formats passages for the LLM. Both arms return the metadata, so they stay identical.

## Part 4. Embedding model and dimensions

**Model: Titan Text Embeddings V2** (`amazon.titan-embed-text-v2:0`). Both services support it, it's what notebook 01 used, and v1 can call it with the same parameters KB uses.

**Dimensions:** the corpus is about 80 chunks, so storage and search cost are negligible at any size. Rule: use **1024** unless a smaller size is tied within noise *and* there's a reason to save space (there isn't one here). The table below just checks for surprises.

In [9]:
dims_table = pd.DataFrame({
    dims: metrics_at(run(SemanticIndex(chunk_sets[CHUNKING], dims).search), 3)
    for dims in (256, 512, 1024)
}).T.rename_axis("dims")
dims_table.columns = [c.replace("@k", "@3") for c in dims_table.columns]
dims_table

,hit@1,hit@3,recall@3,mrr,ctx_tokens@3
dims,,,,,
256,0.944,1.0,1.0,0.972,231.111
512,0.944,1.0,1.0,0.972,228.556
1024,1.000,1.0,1.0,1.000,248.500


In [10]:
EMBED_DIMS = 1024

## Part 5. Retrieval mode

**Semantic (cosine k-NN)**, with no reranking. This one isn't measured; it follows from the constraints:

- v2 on S3 Vectors only supports semantic search.
- Even on a hybrid-capable store, KB's hybrid fusion can't be configured, so v1 could not reproduce it.
- Notebook 01 found hybrid gave no gain with `markdown` chunks (semantic MRR 1.000 vs hybrid 0.972), so on this corpus we lose nothing measurable.

Both arms use **cosine** distance: `space_type: cosinesimil` on the OpenSearch `knn_vector` field, and `distanceMetric: cosine` on the S3 Vectors index.

## Part 6. top-k

top-k is how many chunks the agent receives. A larger k gives more chances to include the right passage (and both docs for comparison questions) but puts more tokens in the prompt, which raises cost and noise.

**Decision rule:** the smallest k where **recall@k ≥ 0.95**, with a floor of 3. The floor is there because comparison questions ("which product…") need passages from more than one document, and the golden set will contain more of them than this seed set does.

In [11]:
rows = run(SemanticIndex(chunk_sets[CHUNKING], EMBED_DIMS).search, k=10)
topk_table = pd.DataFrame({k: metrics_at(rows, k)[["hit@k", "recall@k", "ctx_tokens@k"]] for k in range(1, 9)}).T.rename_axis("k")
topk_table

,hit@k,recall@k,ctx_tokens@k
k,,,
1,1.0,0.944,111.556
2,1.0,1.000,170.222
3,1.0,1.000,248.500
4,1.0,1.000,297.333
5,1.0,1.000,358.611
6,1.0,1.000,429.167
7,1.0,1.000,496.667
8,1.0,1.000,564.111


In [12]:
TOP_K = max(3, int(topk_table.index[topk_table["recall@k"] >= 0.95].min()))
print(f"top-k = {TOP_K}  (~{topk_table.loc[TOP_K, 'ctx_tokens@k']:.0f} context tokens per query)")

top-k = 3  (~248 context tokens per query)


## Part 7. What still differs between the arms

Some differences come with the platform and can't be removed. They are **part of the factor**, so write them down: if a result differs, these are the first suspects.

| Difference | v1 | v2 | Can it affect quality? |
| --- | --- | --- | --- |
| Vector index | Lucene HNSW (approximate) | S3 Vectors' own index (approximate) | Slightly, by missing a near neighbour. With ~80 vectors it should be negligible |
| Who embeds the chunks | Our ingestion script | KB sync | Only if the text sent differs (see parity check) |
| Who embeds the query | `rag_client` → Bedrock | KB, inside `Retrieve` | No, same model and dims. But latency is measured differently: v2 includes it in one call |
| Document parsing | We read the file as-is | KB's parser for `.md` files | Possibly: verify that the stored chunk text equals the file text |
| Score scale | OpenSearch score | KB score | No, compare by rank, never by raw score |
| Network path | Client → OpenSearch domain | Client → Bedrock API → S3 Vectors | Latency only |

### Parity check (do this first on Day 3, before measuring)

1. Ingest the same chunk files into both arms.
2. For each eval question, retrieve the top `TOP_K` from both and compare the **`chunk_id` sets** (Jaccard overlap = |A∩B| / |A∪B|).
3. Expect a mean overlap ≥ 0.9 and the same top-1 on almost every question. Check every question where top-1 differs: open the stored chunk text in v2 and diff it against the file.

Only after parity holds do quality, latency and cost numbers mean anything.

### Preview: the chunk files both arms ingest

v1's ingestion indexes these chunks directly. v2's data source points at an S3 prefix holding one `.md` file per chunk plus its metadata sidecar. The cell below writes them locally (gitignored cache) so you can see the exact shape.

In [13]:
def export_chunks(chunks: list[dict], out_dir: Path) -> list[Path]:
    """One file per chunk + a KB metadata sidecar. '#' isn't safe in S3 keys, so chunk_id 'faq#3' -> 'faq__3.md'."""
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for c in chunks:
        path = out_dir / (c["chunk_id"].replace("#", "__") + ".md")
        path.write_text(c["text"], encoding="utf-8")
        meta = {"metadataAttributes": {k: c[k] for k in ("doc_id", "chunk_id", "title", "section")}}
        Path(f"{path}.metadata.json").write_text(json.dumps(meta, ensure_ascii=False), encoding="utf-8")
        paths.append(path)
    return paths


EXPORT_DIR = CACHE_DIR / "kb_chunks" / CHUNKING
paths = export_chunks(chunk_sets[CHUNKING], EXPORT_DIR)
example = next(p for p in paths if p.name == "repair-bond-oil__4.md")
print(f"{len(paths)} chunk files in {EXPORT_DIR.relative_to(ROOT)}\n")
print(f"--- {example.name}\n{example.read_text()}\n")
print(f"--- {example.name}.metadata.json\n{Path(f'{example}.metadata.json').read_text()}")

76 chunk files in notebooks/.cache/kb_chunks/markdown

--- repair-bond-oil__4.md
## How to use
Apply 2–3 drops to damp mid-lengths and ends before styling, or smooth a drop over dry hair as a finishing shine. A little goes a long way.

--- repair-bond-oil__4.md.metadata.json
{"metadataAttributes": {"doc_id": "repair-bond-oil", "chunk_id": "repair-bond-oil#4", "title": "Repair Bond Oil", "section": "How to use"}}


## Part 8. Freeze the spec

Everything above goes into one file that v1 ingestion, v2 Terraform and the eval runner all read. Once Day 3 measurements start, **don't edit it**. If something has to change, bump `version` and rerun both arms, so results from different specs are never compared.

In [14]:
EXPERIMENT = {
    "name": "rag-v1-vs-v2",
    "version": 1,
    "created": date.today().isoformat(),
    "source": "notebooks/02_rag_experiment_design.ipynb",
    "hypothesis": (
        "With identical chunks, embeddings, retrieval mode and top-k, v1 and v2 reach the same retrieval "
        "quality within noise (|delta MRR| < one question). v2 costs less per month and needs less "
        "operational work. Latencies differ by less than 100 ms at p50."
    ),
    "factor": {
        "name": "retrieval platform",
        "levels": {
            "v1": "Amazon OpenSearch Service managed domain; our ingestion pipeline embeds and indexes chunks",
            "v2": "Amazon Bedrock Knowledge Base on an Amazon S3 Vectors store; KB sync embeds and indexes chunks",
        },
        "baseline": "v1",
    },
    "controlled": {
        "corpus": "data/knowledge_base/ (same git commit for both arms)",
        "chunking": {
            "strategy": CHUNKING,
            "implementation": "our chunker writes one file per chunk; KB data source uses chunkingStrategy NONE",
            "metadata_fields": ["doc_id", "chunk_id", "title", "section"],
            "prompt_format": "rag_client prepends '<title> | <section>' to each passage it gives the LLM",
        },
        "embedding": {"model_id": EMBED_MODEL_ID, "dimensions": EMBED_DIMS, "normalize": True},
        "retrieval": {"mode": "semantic", "distance": "cosine", "top_k": TOP_K, "reranking": None, "filters": None},
        "generation": {
            "model_id": None,  # TODO Day 3: same Bedrock model for both arms
            "temperature": 0,
            "prompt": None,  # TODO Day 3: path under services/agents/prompts/
        },
        "eval_set": "data/eval/ golden set (Day 3)",
    },
    "parity_check": {"metric": "mean Jaccard overlap of top-k chunk_ids", "pass_if": ">= 0.9"},
    "metrics": {
        "retrieval": ["hit@1", f"hit@{TOP_K}", f"recall@{TOP_K}", "mrr"],
        "answer": ["faithfulness", "answer_relevancy"],
        "latency": ["p50_ms", "p95_ms"],
        "cost": ["cost_per_query_usd", "fixed_monthly_usd"],
    },
    "decision_rule": "docs/architecture.md, RAG evaluation plan",
}

out = ROOT / "evals" / "rag" / "experiment_config.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(EXPERIMENT, indent=2, ensure_ascii=False) + "\n")
print(f"wrote {out.relative_to(ROOT)}\n")
print(json.dumps(EXPERIMENT["controlled"], indent=2))

wrote evals/rag/experiment_config.json

{
  "corpus": "data/knowledge_base/ (same git commit for both arms)",
  "chunking": {
    "strategy": "markdown",
    "implementation": "our chunker writes one file per chunk; KB data source uses chunkingStrategy NONE",
    "metadata_fields": [
      "doc_id",
      "chunk_id",
      "title",
      "section"
    ],
    "prompt_format": "rag_client prepends '<title> | <section>' to each passage it gives the LLM"
  },
  "embedding": {
    "model_id": "amazon.titan-embed-text-v2:0",
    "dimensions": 1024,
    "normalize": true
  },
  "retrieval": {
    "mode": "semantic",
    "distance": "cosine",
    "top_k": 3,
    "reranking": null,
    "filters": null
  },
  "generation": {
    "model_id": null,
    "temperature": 0,
    "prompt": null
  },
  "eval_set": "data/eval/ golden set (Day 3)"
}


## Summary

| Parameter | Value | Why |
| --- | --- | --- |
| Independent variable | Retrieval platform: v1 OpenSearch vs v2 Bedrock KB | The thing the project wants to compare |
| v2 vector store | Amazon S3 Vectors | Pay per use, and a genuinely different backend from v1 |
| Chunking | `markdown` (one chunk per `##` section), done in our code; KB uses `NONE` | Only way to give both arms identical chunks; also scored highest (Part 3) |
| Product name for the LLM | `title` and `section` as metadata, prepended by `rag_client` | Fixes `markdown`'s missing product name without changing embeddings |
| Embedding | Titan Text Embeddings V2, 1024 dims, normalised | Supported by both; size doesn't matter at this corpus size |
| Retrieval mode | Semantic (cosine k-NN), no reranking | S3 Vectors is semantic-only, KB hybrid isn't configurable, hybrid gave no gain in 01 |
| top-k | 3 | Smallest k with recall ≥ 0.95 is 2; the floor of 3 leaves room for comparison questions (~250 context tokens) |
| Expected outcome | Same quality within noise; the comparison is decided by cost, latency and operational effort | Everything that drives retrieval quality is held constant |

**Follow-ups outside this notebook:**
- Update the RAG section of `docs/architecture.md`: v2's vector store (S3 Vectors, no longer "assumed"), the controlled-variables table, and the parity check.
- Day 3: write the golden set, run the parity check, then measure.